# PACT-18 两阶段 LoRA · Kaggle 冒烟验证

目的：**在真实 T4×2 上把新代码路径完整跑一遍**，只验证执行正确性，不追求分数。

覆盖的技术点：

1. 8B 权重按 `num_hidden_layers=24` 截断加载（只实例化前 24 个 block）
2. 4-bit NF4 + LoRA（r16，7 个投影）
3. stage 1：只用外部语料，三元组（同 prompt、3 个不同模型）
4. stage 2：只用主办方 2,699 条中的 4 个 prompt，按 prompt 分组 2 折
5. 双头损失 `CE(plain) + CE(centred)`
6. 两卡数据并行（梯度 all-reduce）
7. 经典成员 → 装配 → `submission.parquet`

数据是**缩小的**：4 个 prompt × 18 段 + 40 个外部组。预计 8–15 分钟。


In [ ]:
import glob, os, sys, time
import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, '| cuda', torch.version.cuda)
print('devices :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | cc {p.major}.{p.minor}')
print('input   :', sorted(os.listdir('/kaggle/input')))


In [ ]:
import glob, os, sys, time
import json

import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, '| cuda', torch.version.cuda)
print('devices :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | cc {p.major}.{p.minor}')
print('input   :', sorted(os.listdir('/kaggle/input')))


def pick_dir(patterns, marker):
    for pat in patterns:
        for cand in sorted(glob.glob(pat)):
            if os.path.isdir(cand) and os.path.exists(os.path.join(cand, marker)):
                return cand
    return None

DS = pick_dir(['/kaggle/input/datasets/*/ntl-pact18-verify',
               '/kaggle/input/ntl-pact18-verify',
               '/kaggle/input/*/ntl-pact18-verify'], 'ntl_merged_external_large.parquet')
COMP = pick_dir(['/kaggle/input/competitions/name-that-llm',
                 '/kaggle/input/name-that-llm'], 'test.csv')

# Model source qwen-lm/qwen-3/Transformers/8b-base/1. Explicit single-level
# paths first: each '*' matches one symlinked mount level (a recursive '**'
# scan does not reliably descend into the mount symlinks). The recursive scan
# stays as the fallback - it is what the public 0.55x kernels use.
MODEL = pick_dir([
    '/kaggle/input/models/qwen-lm/qwen-3/Transformers/8b-base/1',
    '/kaggle/input/models/*/qwen-3/Transformers/8b-base/1',
    '/kaggle/input/models/*/*/Transformers/8b-base/1',
    '/kaggle/input/qwen-3',
    '/kaggle/input/*/qwen-3',
], 'config.json')
if MODEL is None:
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            meta = json.loads(open(cfg, encoding='utf-8').read())
        except Exception:
            continue
        if str(meta.get('model_type', '')).startswith('qwen3') and int(meta.get('hidden_size', 0)) >= 4096:
            MODEL = os.path.dirname(cfg)
            break
print('DS   :', DS)
print('COMP :', COMP)
print('MODEL:', MODEL)
assert DS and COMP and MODEL, 'Add Input: competition + ntl-pact18-verify + qwen-lm/qwen-3 (Transformers/8b-base/1)'
print('dataset files:', sorted(os.listdir(DS))[:8])

In [ ]:
!pip install -q --no-index --find-links {DS} bitsandbytes 2>&1 | tail -2
import bitsandbytes as bnb
print('bitsandbytes', bnb.__version__)
import transformers
print('transformers', transformers.__version__)


In [ ]:
import shutil, subprocess

WORK = '/kaggle/working/ntl-pact18-code'
if not os.path.isdir(WORK):
    zip_path = f'{DS}/ntl-pact18-code.zip'
    if os.path.exists(zip_path):
        subprocess.call(f'cd /kaggle/working && unzip -oq {zip_path}', shell=True)
    else:
        hits = glob.glob(f'{DS}/**/src/ntl_pact18/pipeline.py', recursive=True)
        assert hits, f'no package under {DS}'
        shutil.copytree(hits[0].split('/src/ntl_pact18/')[0], WORK)
os.chdir(WORK)
sys.path.insert(0, f'{WORK}/src')
print('cwd', os.getcwd())
print('configs:', sorted(os.listdir('configs'))[:6], '...')


In [ ]:
# fast unit subset: the new triplet plan and the parallel primitives
!PYTHONPATH={WORK}/src python -m pytest -q tests/test_two_stage.py tests/test_dist_utils.py 2>&1 | tail -4


In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv(f'{COMP}/train.csv')
test = pd.read_csv(f'{COMP}/test.csv')

# 4 host prompts x 18 models (grouped folds need whole prompts)
keep_prompts = sorted(train['prompt_id'].unique())[:4]
host = train[train['prompt_id'].isin(keep_prompts)].copy()
host['origin'] = 'competition'

# 40 pool groups that contain >= 3 distinct models, so stage 1 gets triplets.
# Take up to 6 *distinct-model* rows per prompt (head(4) could return four
# passages by the same model, which forms no triplet).
pool = pd.read_parquet(f'{DS}/ntl_merged_external_large.parquet')
sizes = pool.groupby('prompt_id')['model'].nunique()
eligible = sizes[sizes >= 3].index[:40]
small_pool = (pool[pool['prompt_id'].isin(eligible)]
              .sort_values('id')
              .drop_duplicates(['prompt_id', 'model'])
              .groupby('prompt_id', group_keys=False)
              .head(6)
              .copy())
small_pool['origin'] = 'external'

tiny = pd.concat([host, small_pool], ignore_index=True)
cols = ['id', 'prompt_id', 'text', 'model', 'origin']
tiny = tiny[cols].drop_duplicates('id').reset_index(drop=True)
tiny.to_csv('/kaggle/working/train_smoke.csv', index=False)
print('rows', len(tiny), '| host', (tiny.origin == 'competition').sum(),
      '| pool', (tiny.origin == 'external').sum())
print('pool groups', tiny[tiny.origin == 'external']['prompt_id'].nunique(),
      '| host prompts', tiny[tiny.origin == 'competition']['prompt_id'].nunique())
print('test rows', len(test))


In [ ]:
cmd = f'''python scripts/train_parallel.py --stage all \\
  --config configs/kaggle_2stage_8b.yaml \\
  --train-file /kaggle/working/train_smoke.csv \\
  --test-file {COMP}/test.csv \\
  --submission /kaggle/working/submission.parquet \\
  --deadline-hours 0.7 --reserve-hours 0.15 \\
  --set features.qwen.model_id={MODEL} \\
  --set data.n_folds=2 \\
  --set features.qwen.stage1_epochs=1 \\
  --set features.qwen.stage2_epochs=1 \\
  --set features.qwen.stage1_time_budget_hours=0.20 \\
  --set features.qwen.time_budget_hours=0.12 \\
  --set features.qwen.log_every=5 \\
  --set classical.max_fit_rows=0'''

t0 = time.time()
rc = subprocess.call(cmd, shell=True)
print('\\n[exit]', rc, 'in', round((time.time() - t0) / 60, 1), 'min')
assert rc == 0, 'training failed - inspect artifacts/parallel/logs/qwen_rank0.log'


In [ ]:
import glob, json

for f in sorted(glob.glob('artifacts/parallel/logs/*.log')):
    print('---', f)
    lines = open(f, encoding='utf-8', errors='ignore').readlines()
    print(''.join(lines[-25:]))

meta_path = 'artifacts/parallel/qwen_oof.json'
if os.path.exists(meta_path):
    print('\n=== qwen stage metadata ===')
    print(json.dumps(json.load(open(meta_path)), indent=2))


In [ ]:
import pyarrow.parquet as pq
from ntl_pact18.labels import LABELS

sub = pd.read_parquet('/kaggle/working/submission.parquet')
print('shape      :', sub.shape)
print('columns    :', list(sub.columns))
print('schema     :', pq.read_schema('/kaggle/working/submission.parquet').names)
print('nulls      :', int(sub.isna().sum().sum()), '| dup ids:', int(sub['id'].duplicated().sum()))
print('labels ok  :', set(sub['model']).issubset(set(LABELS)))
print('rows match :', len(sub) == len(test))
assert len(sub) == len(test) and set(sub['model']).issubset(set(LABELS))
print(sub.to_string(index=False))


## 通过标准

| 检查 | 期望 |
|---|---|
| `truncated trunk: 36 -> 24 blocks` | 出现，说明截断生效 |
| `stage1: ... triplets` | 出现，且 `loss` 有限 |
| `stage2: train=72 ...` | 两折各 36 行 |
| `[exit] 0` | 训练进程正常退出 |
| `submission.parquet` | 3 行、两列、标签合法 |
| `trainable_parameters` | LoRA 约 23M + 头 1.2M |

任何一步 `assert` 失败，先看 `qwen_rank0.log` 的最后 25 行。
